In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [33]:
import re 
def clean_text(text):
    # 1. Replace escaped newlines and line breaks with spaces
    text = text.replace('\\n', ' ').replace('\n', ' ')
    
    # 2. Remove Markdown headers (e.g., #, ##, ###)
    text = re.sub(r'#+\s*', '', text)
    
    # 3. Collapse multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text)
    
    # 4. Strip leading and trailing whitespace
    return text.strip()


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if "GEMINI_API_KEY" in os.environ and "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

In [20]:
model = ChatGoogleGenerativeAI(
    model="models/gemini-3.1-flash-lite",
    temperature=0.2
   
)

In [21]:
class BlogState(TypedDict):

    title:str
    outline:str
    content: str



In [22]:
title="Islamic Finance: Principles, Practices, and Global Impact"
prompt = f"""
    Create a detailed outline for the following blog on the topic : {title}
    """

model.invoke(prompt).content[0]["text"]

'This outline is designed to provide a comprehensive, structured, and engaging overview of Islamic Finance, suitable for a professional blog post.\n\n---\n\n# Blog Outline: Islamic Finance: Principles, Practices, and Global Impact\n\n## I. Introduction\n*   **Hook:** Briefly define Islamic Finance not just as a religious requirement, but as a growing, ethical alternative to conventional banking.\n*   **The "Why":** Mention the rapid growth of the industry (trillions in assets) and its increasing relevance in global markets.\n*   **Thesis Statement:** Islamic Finance offers a value-based economic framework that prioritizes social justice, risk-sharing, and tangible asset backing.\n\n## II. The Core Principles (The "Sharia" Foundation)\n*   **Prohibition of Riba (Interest):** Explain why money is considered a medium of exchange, not a commodity that earns profit on its own.\n*   **Prohibition of Gharar (Uncertainty/Speculation):** Why excessive risk-taking and ambiguity in contracts are 

In [23]:
def create_outilne(state:BlogState) -> BlogState:

    title=state['title']


    prompt = f"""
    Create a detailed outline for the following blog on the topic : {title}
    """
    response =model.invoke(prompt).content[0]["text"]
    state['outline'] = response
    return state

In [24]:
def create_content(state:BlogState) ->BlogState:
    title=state['title']
    outline=state['outline']

    prompt = f"""
    Create a detailed blog content for the following blog on the topic : {title} with the following outline : {outline}
    """
    response =model.invoke(prompt).content[0]["text"]
    state['content'] = response
    return state

In [25]:
graph=StateGraph(BlogState)

graph.add_node('create_utline',create_outilne)
graph.add_node('create_content',create_content)

graph.add_edge(START,'create_utline')
graph.add_edge('create_utline','create_content')
graph.add_edge('create_content',END)
workflow=graph.compile()

In [27]:
output=workflow.invoke({'title':"Islamic Finance: Principles, Practices, and Global Impact"})

In [28]:
output

{'title': 'Islamic Finance: Principles, Practices, and Global Impact',
 'outline': 'This outline is designed to provide a comprehensive, structured, and engaging overview of Islamic Finance, suitable for a professional blog post.\n\n---\n\n# Blog Outline: Islamic Finance: Principles, Practices, and Global Impact\n\n## I. Introduction\n*   **Hook:** Briefly define Islamic Finance not just as a religious requirement, but as a growing, ethical alternative to conventional banking.\n*   **The "Why":** Mention the shift in global interest toward socially responsible investing (SRI) and ESG (Environmental, Social, and Governance) criteria.\n*   **Thesis Statement:** Islamic finance offers a unique framework based on risk-sharing and moral integrity, which is increasingly influencing the global financial landscape.\n\n## II. Core Principles: The Ethical Foundation\n*   **Prohibition of Riba (Interest):** Explain the concept of money as a medium of exchange, not a commodity that generates profi

In [30]:
output['title']

'Islamic Finance: Principles, Practices, and Global Impact'

In [34]:
clean_text(output['content'])

'Islamic Finance: Principles, Practices, and Global Impact In an era where global markets are increasingly scrutinized for their social and environmental impact, a centuries-old financial framework is stepping into the spotlight. Islamic Finance is no longer just a religious requirement for Muslims; it has emerged as a robust, ethical alternative to conventional banking. As the world pivots toward Socially Responsible Investing (SRI) and ESG (Environmental, Social, and Governance) criteria, Islamic finance offers a compelling blueprint. By prioritizing risk-sharing, moral integrity, and tangible economic activity, this system is not only surviving—it is thriving as a pillar of the modern global financial landscape. --- The Core Principles: The Ethical Foundation At its heart, Islamic finance is built on the concept of *Maqasid al-Sharia* (the objectives of the law), which seeks to protect faith, life, intellect, lineage, and property. To achieve this, the system operates on five fundam

In [35]:
clean_text(output['content'])

'Islamic Finance: Principles, Practices, and Global Impact In an era where global markets are increasingly scrutinized for their social and environmental impact, a centuries-old financial framework is stepping into the spotlight. Islamic Finance is no longer just a religious requirement for Muslims; it has emerged as a robust, ethical alternative to conventional banking. As the world pivots toward Socially Responsible Investing (SRI) and ESG (Environmental, Social, and Governance) criteria, Islamic finance offers a compelling blueprint. By prioritizing risk-sharing, moral integrity, and tangible economic activity, this system is not only surviving—it is thriving as a pillar of the modern global financial landscape. --- The Core Principles: The Ethical Foundation At its heart, Islamic finance is built on the concept of *Maqasid al-Sharia* (the objectives of the law), which seeks to protect faith, life, intellect, lineage, and property. To achieve this, the system operates on five fundam